# Ingest RAND Longitudinal SPSS Files to Databricks

**Source:** `D:\OneDrive\Documents\Databricks Training\HRS\RAND Longitudinal\randhrs1992_2022v1_SPSS`  

**Destination Volume:** `dev_catalog.brz_landing_hrs.hrs_vol`  
---

## Important Note
Databricks notebooks run on cloud compute and **cannot directly access your local PC files**. You must first upload files from your local directory to Databricks.


### UI Upload (Easiest and the method I used)

1. Download the randhrs1992_2020v1_spss file from the HRS website.
2. Open **Catalog Explorer** → `dev_catalog` → `brz_landing_hrs` → `hrs_vol`
3. Click **Upload** button
4. Select `.sav` file(s) from your local directory
5. Wait for upload to complete
---


In [0]:
# List files in the volume to verify upload
import os

volume_path = "/Volumes/dev_catalog/brz_landing_hrs/hrs_vol"

print("Files in volume:")
print("=" * 80)

files = dbutils.fs.ls(volume_path)
for file_info in files:
    size_mb = file_info.size / (1024 * 1024)
    print(f"{file_info.name:50s} {size_mb:>10.2f} MB")

print(f"\nTotal files: {len(files)}")

In [0]:
%pip install pyreadstat

In [0]:
# Read SPSS (.sav) file and convert to Delta table
import pyreadstat
import pandas as pd
from pyspark.sql import SparkSession

# Configuration
sav_file_path = "/Volumes/dev_catalog/brz_landing_hrs/hrs_vol/randhrs1992_2022v1.sav"
table_name = "dev_catalog.brz_raw_hrs.randhrs1992_2022v1"

print(f"Reading SPSS file: {sav_file_path}")
print("This may take a few minutes for large files...\n")

# Read SPSS file using pyreadstat
df_pandas, meta = pyreadstat.read_sav(f"/dbfs{sav_file_path}")

print(f"✓ Read {len(df_pandas):,} records")
print(f"✓ Columns: {len(df_pandas.columns)}")
print(f"\nFirst few columns: {list(df_pandas.columns[:10])}")

# Convert to Spark DataFrame
df_spark = spark.createDataFrame(df_pandas)

print(f"\nWriting to Delta table: {table_name}")
df_spark.write.mode("overwrite").saveAsTable(table_name)

print(f"\n✓ Successfully created table: {table_name}")
print(f"✓ Total records: {df_spark.count():,}")

In [0]:
%sql
-- Verify the table was created and preview data
SELECT * FROM dev_catalog.brz_raw_hrs.randhrs1992_2022v1 LIMIT 10